# Lab 04-04: tool calling

Lab: `corso/laboratori/lab-04-04-tool-calling`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Server and agent

The agent asks the model to call the `get_weather` function; the script runs it (fake or real weather) and returns the result to the model.

In [ ]:
import subprocess, time, urllib.request

def avvia_server(args, porta=8080, attesa=300):
    """Start `llama serve` in the background and wait for /health."""
    log = open("/tmp/llama-server.log", "w")
    p = subprocess.Popen(["llama", "serve", *args.split(), "--host", "127.0.0.1", "--port", str(porta)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(attesa):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{porta}/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    raise RuntimeError("the server does not answer: " + open("/tmp/llama-server.log").read()[-1500:])

def ferma_server(p):
    p.terminate(); p.wait()

In [ ]:
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 1 -a qwen-0.8b -t %d" % T)

In [ ]:
!pip -q install openai

In [ ]:
%%bash
cd $LABS/lab-04-04-tool-calling
python3 agente_meteo.py

With `--reale` the function queries open-meteo.com (free, no key).

In [ ]:
%%bash
cd $LABS/lab-04-04-tool-calling
python3 agente_meteo.py --reale "Che tempo fa a Palermo e a Torino?"

In [ ]:
ferma_server(p)